# Visao geral do pipeline climatico

Este notebook usa um ano de dados ficticios para demonstrar uma preparacao minima antes de qualquer etapa de modelagem. O foco e mostrar estrutura, falhas observacionais, rastreabilidade e padronizacao.

In [ ]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inmet_path = RAW_DIR / "exemplo_inmet_ficticio.csv"
era5_path = RAW_DIR / "exemplo_era5_ficticio.csv"
indice_horario = pd.date_range("2024-01-01 00:00", "2024-12-31 23:00", freq="h")

## Leitura dos dados ficticios

In [ ]:
inmet = pd.read_csv(inmet_path, parse_dates=["data_hora"])
era5 = pd.read_csv(era5_path, parse_dates=["data_hora"])

print(f"INMET ficticio: {len(inmet)} linhas brutas")
print(f"ERA5 ficticio: {len(era5)} linhas brutas")

display(inmet.head())
display(era5.head())

## Checagem de colunas

In [ ]:
colunas_inmet = {
    "data_hora",
    "fonte",
    "estacao",
    "temperatura_c",
    "umidade_relativa_pct",
    "precipitacao_mm",
}

colunas_era5 = {
    "data_hora",
    "fonte",
    "ponto_grade",
    "temperatura_2m_k",
    "precipitacao_total_m",
}

assert colunas_inmet.issubset(inmet.columns)
assert colunas_era5.issubset(era5.columns)
assert not inmet["data_hora"].duplicated().any()
assert not era5["data_hora"].duplicated().any()

print("Estruturas minimas conferidas.")

## Padronizacao de datas e comparacao de resolucao temporal

In [ ]:
freq_inmet = inmet["data_hora"].sort_values().diff().dropna().mode().iloc[0]
freq_era5 = era5["data_hora"].sort_values().diff().dropna().mode().iloc[0]

pd.DataFrame(
    {
        "fonte": ["INMET_EXEMPLO", "ERA5_EXEMPLO"],
        "linhas_brutas": [len(inmet), len(era5)],
        "resolucao_inferida": [freq_inmet, freq_era5],
        "inicio": [inmet["data_hora"].min(), era5["data_hora"].min()],
        "fim": [inmet["data_hora"].max(), era5["data_hora"].max()],
    }
)

## Falhas no INMET ficticio

O exemplo INMET cobre o mesmo ano, mas simula problemas comuns de dados observacionais: horarios ausentes e valores vazios em algumas variaveis.

In [ ]:
esperado_horario = pd.DataFrame({"data_hora": indice_horario})
horarios_ausentes = esperado_horario.merge(inmet[["data_hora"]], on="data_hora", how="left", indicator=True)
horarios_ausentes = horarios_ausentes.loc[horarios_ausentes["_merge"].eq("left_only"), ["data_hora"]]

resumo_falhas = pd.DataFrame(
    {
        "item": [
            "horas esperadas no ano",
            "linhas INMET observadas",
            "horarios INMET ausentes",
            "valores vazios no INMET",
        ],
        "quantidade": [
            len(indice_horario),
            len(inmet),
            len(horarios_ausentes),
            int(inmet.isna().sum().sum()),
        ],
    }
)

display(resumo_falhas)
display(horarios_ausentes.head(10))

## Insercao de horarios ausentes e preenchimento de lacunas

A grade horaria completa e inserida. Lacunas curtas nas variaveis INMET sao preenchidas por interpolacao temporal, mantendo flags para indicar o que foi imputado.

In [ ]:
inmet_base = inmet.rename(columns={"temperatura_c": "temperatura_inmet_c"}).set_index("data_hora").sort_index()
inmet_horario = inmet_base.reindex(indice_horario)
colunas_numericas = ["temperatura_inmet_c", "umidade_relativa_pct", "precipitacao_mm"]

for coluna in colunas_numericas:
    inmet_horario[f"{coluna}_imputado"] = inmet_horario[coluna].isna()

inmet_horario["registro_inmet_inserido"] = inmet_horario["fonte"].isna()
inmet_horario["fonte"] = inmet_horario["fonte"].fillna("INMET_EXEMPLO")
inmet_horario["estacao"] = inmet_horario["estacao"].fillna("ESTACAO_FICTICIA")
inmet_horario[colunas_numericas] = inmet_horario[colunas_numericas].interpolate(method="time", limit=6, limit_direction="both")
inmet_horario["precipitacao_mm"] = inmet_horario["precipitacao_mm"].fillna(0)
inmet_horario["umidade_relativa_pct"] = inmet_horario["umidade_relativa_pct"].clip(lower=0, upper=100)
inmet_horario.index.name = "data_hora"
inmet_horario = inmet_horario.reset_index()

inmet_horario[["temperatura_inmet_c_imputado", "umidade_relativa_pct_imputado", "precipitacao_mm_imputado", "registro_inmet_inserido"]].sum()

## Padronizacao de unidades do ERA5 ficticio

In [ ]:
era5_padronizado = era5.copy()
era5_padronizado["temperatura_era5_c"] = era5_padronizado["temperatura_2m_k"] - 273.15
era5_padronizado["precipitacao_era5_mm"] = era5_padronizado["precipitacao_total_m"] * 1000
era5_horario = era5_padronizado.set_index("data_hora")[["temperatura_era5_c", "precipitacao_era5_mm"]].reindex(indice_horario)
era5_horario["registro_era5_interpolado"] = era5_horario["temperatura_era5_c"].isna()
era5_horario[["temperatura_era5_c", "precipitacao_era5_mm"]] = era5_horario[["temperatura_era5_c", "precipitacao_era5_mm"]].interpolate(method="time", limit_direction="both")
era5_horario.index.name = "data_hora"
era5_horario = era5_horario.reset_index()

era5_horario.head()

## Base processada ficticia

A base final junta o INMET ja preenchido com o ERA5 convertido para a mesma grade horaria. As flags permanecem para manter rastreabilidade.

In [ ]:
base_processada = inmet_horario.merge(era5_horario, on="data_hora", how="inner")
base_processada = base_processada[
    [
        "data_hora",
        "temperatura_inmet_c",
        "umidade_relativa_pct",
        "precipitacao_mm",
        "temperatura_inmet_c_imputado",
        "umidade_relativa_pct_imputado",
        "precipitacao_mm_imputado",
        "registro_inmet_inserido",
        "temperatura_era5_c",
        "precipitacao_era5_mm",
        "registro_era5_interpolado",
    ]
]

print(base_processada.shape)
display(base_processada.head())

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
saida = PROCESSED_DIR / "base_climatica_ficticia_processada.csv"
base_processada.to_csv(saida, index=False)
print(f"Base ficticia salva em: {saida}")